## Usefull libraries:

In [378]:
import numpy as np
import glob
import camb
from camb import model, initialpower
import math
import matplotlib.pyplot as plt
from scipy import special
from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord
from scipy.integrate import quad
from scipy.interpolate import interp1d
from scipy.integrate import simpson
from scipy.integrate import trapezoid
from CosmoFunc import *
from BF_OptMC import *
from lu_functions import *
import time
import os
from numba import jit, prange

## Extract the R_matrix from the .txt file

In [379]:
# paramètres:
N = 2500 # nombre de galaxies qu'on prend


# Pattern pour capturer tous les fichiers avec ce format
fichiers = sorted(glob.glob(f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_{N}g_sample_*.txt'))

# Extraire toutes les matrices
R = []
for fichier in fichiers:
    mat = np.loadtxt(fichier, skiprows=2)
    R.append(mat)
    print(f"✓ Chargé: {fichier} -> forme {mat.shape}")

# Stocker dans des variables R1, R2, etc.
for i, mat in enumerate(R, 1):
    globals()[f'R{i}'] = mat


✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_1.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_10.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_3.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_4.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_6.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_7.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_8.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_9.txt -> forme (9, 9)


##  Statistics on R

In [380]:
# Calculer la matrice moyenne (covariance moyenne)
R_mean = np.mean(R, axis=0)

# Vous pouvez aussi calculer l'écart-type entre les runs
R_std = np.std(R, axis=0)

# Erreur standard de la moyenne (incertitude sur la moyenne)
R_sem = R_std / np.sqrt(len(R))  # 10 runs


# R_thorie est la resulats de Fei:

R_theorie = np.array([
    [5248.882, -58.604, -2725.658, -1.093, 1.034, -9.032, -0.357, 0.153, 12.569],
    [-58.604, 7048.148, 1998.970, 3.336, 0.974, 0.819, 1.109, -13.604, -9.018],
    [-2725.658, 1998.970, 45434.716, 30.027, 3.378, 18.922, 17.950, -0.134, -203.080],
    [-1.093, 3.336, 30.027, 0.140, 0.005, 0.012, 0.061, -0.005, -0.093],
    [1.034, 0.974, 3.378, 0.005, 0.036, -0.002, 0.005, 0.003, -0.016],
    [-9.032, 0.819, 18.922, 0.012, -0.002, 0.085, 0.014, 0.003, -0.085],
    [-0.357, 1.109, 17.950, 0.061, 0.005, 0.014, 0.152, 0.003, -0.042],
    [0.153, -13.604, -0.134, -0.005, 0.003, 0.003, 0.003, 0.098, -0.003],
    [12.569, -9.018, -203.080, -0.093, -0.016, -0.085, -0.042, 0.003, 1.103]])

# Calculer la différence normalisée par l'erreur
difference_normalisee = (R_theorie - R_mean) / R_sem


## Enregistrer la matrice moyenne/ ecart-type et celle de Fei

In [381]:
with open(f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_random_moy_{N}.txt', 'w') as f:
    f.write(f"R matrix mean with 10 samples with {N} galaxies each :\n\n")
    for row in R_mean:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")
    f.write(f"R matrix ecart-type : \n\n")
    for row in R_std:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")  
    f.write(f"R matrix standard error on the mean :\n\n")        
    for row in R_sem:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")  
    f.write("(R_Fei - R_mean) / R_error :\n\n")
    for row in difference_normalisee:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")       

print(f"Results saved in: {f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_random_moy_{N}.txt'}")


with open(f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_exact_Fei.txt', 'w') as f:
    f.write(f"R matrix exact from Fei Qin :\n\n")
    for row in R_theorie:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")
        
print(f"Results saved in:{f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_exact_Fei.txt'}")


Results saved in: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_random_moy_2500.txt
Results saved in:/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_exact_Fei.txt


## Extraire la prédiction théorique sur le Bulk flow

In [382]:
# We can extract the Bulk flow from the R matrix with the jacobian matrix:

V_bkf = [0,0,0] # setting the Bulk flow 
B_pq = R_mean[:3, :3] # we take only the first 3 rows and 3 columns of the R matrix since we want to extract only the Bulk flow and not the shear

np.set_printoptions(precision=6, suppress=True)
print('R_pq part of the matrix with the Bulk Flow component:'),
for row in B_pq:
    print("  ".join(f"{x:10.4f}" for x in row))

R_pq part of the matrix with the Bulk Flow component:
 5344.8397      1.4784  -2604.8143
    1.4784   7634.5110   2827.4235
-2604.8143   2827.4235  48052.2749


In [383]:
# extract the theoritical Bulk Flow from the sigma_B: 
# sigma_B est la racine de la trace de la matrice 3*3 extraite de R_pq:
# la prediction de lambda CDM est : B = 0 +- (2/3)**1/2 * sigma_B
trace_B_pq = np.trace(B_pq)
sigma_B = np.sqrt(trace_B_pq)
B_p= np.sqrt(2/3)*sigma_B
print('The theoritical Bulk Flow from LambdaCDM is : 0 +-', sigma_B, 'km/s')
print('Bp = (2/3)**1/2 * sigmaB = ',B_p,'km/s')

The theoritical Bulk Flow from LambdaCDM is : 0 +- 247.0457964376241 km/s
Bp = (2/3)**1/2 * sigmaB =  201.71204812388706 km/s


In [384]:
# the theoritical Bulk Flow for each component :
Bx_CRMS = np.sqrt(B_pq[0,0])
By_CRMS = np.sqrt(B_pq[1,1])
Bz_CRMS = np.sqrt(B_pq[2,2])
print('The theoritical Bulk Flow Bx = 0 +-',Bx_CRMS,'km/s')
print('The theoritical Bulk Flow By = 0 +-',By_CRMS,'km/s')
print('The theoritical Bulk Flow Bz = 0 +-',Bz_CRMS,'km/s')

The theoritical Bulk Flow Bx = 0 +- 73.10841034039791 km/s
The theoritical Bulk Flow By = 0 +- 87.3756889100166 km/s
The theoritical Bulk Flow Bz = 0 +- 219.20829104415733 km/s


## extraire l'amplitude du bulk flow de nos estimateurs

In [385]:
# B = np.sqrt( Bx**2 + By**2 + Bz**2)
# pour l'estimateur etaMLE:
# We import the bulk flow results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results/'
Fit_tech1 = 'etaMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'


# We load the bulk flow results of the data:
Bx_data_eta, By_data_eta, Bz_data_eta = np.loadtxt(Input_dir + f'Bulk_flow_data_{Fit_tech1}.txt',skiprows=6,usecols=1,max_rows=3)
# we load the error on the Bx,By,Bz estimation for the data:
err_Bx_eta, err_By_eta ,err_Bz_eta = np.loadtxt( Input_dir + f'Bulk_flow_data_{Fit_tech1}.txt', skiprows=6, usecols=2, max_rows=3)

B_data_eta = np.sqrt( Bx_data_eta**2 + By_data_eta**2 + Bz_data_eta**2)
# et on fait la propagation des incertitudes sur B:
err_B_eta = (np.abs(Bx_data_eta*err_Bx_eta) + np.abs(By_data_eta*err_By_eta) + np.abs(Bz_data_eta*err_Bz_eta)) / B_data_eta

# load the file with the mocks values of BF
all_mocks_eta = np.loadtxt(Input_dir + f'Bulk_flow_all_mocks_{Fit_tech1}.txt', skiprows=1)
# we extract the values of Bx,By,Bz and the true values of Ux, Uy, Uz for all the mocks:
Bx_mocks_eta = all_mocks_eta[:, 1]  
By_mocks_eta = all_mocks_eta[:, 2] 
Bz_mocks_eta = all_mocks_eta[:, 3]  
Ux_mocks_eta = all_mocks_eta[:, 4]
Uy_mocks_eta = all_mocks_eta[:, 5]
Uz_mocks_eta = all_mocks_eta[:, 6]

Bx_mocks_CRMS_eta = np.std(Bx_mocks_eta, ddof=1)
By_mocks_CRMS_eta = np.std(By_mocks_eta, ddof=1)
Bz_mocks_CRMS_eta = np.std(Bz_mocks_eta, ddof=1)
Ux_mocks_CRMS_eta = np.std(Ux_mocks_eta, ddof=1)
Uy_mocks_CRMS_eta = np.std(Uy_mocks_eta, ddof=1)
Uz_mocks_CRMS_eta = np.std(Uz_mocks_eta, ddof=1)

# 1. Calculez l'amplitude B pour chaque mock
B_mocks_eta = np.sqrt(Bx_mocks_eta**2 + By_mocks_eta**2 + Bz_mocks_eta**2)
U_mocks_eta = np.sqrt(Ux_mocks_eta**2 + Uy_mocks_eta**2 + Uz_mocks_eta**2)
# 2. L'erreur est l'écart-type de cette distribution
err_B_mocks_opt_eta = np.std(B_mocks_eta, ddof=1)
err_U_mocks_true_eta = np.std(U_mocks_eta, ddof=1)

print('For estimator etaMLE the Bulk Flow with propagation formula is:',B_data_eta,'+-',err_B_eta,'km/s')
print('For estimator etaMLE the Bulk Flow with mocks is:',B_data_eta,'+-',err_B_mocks_opt_eta,'km/s')


For estimator etaMLE the Bulk Flow with propagation formula is: 377.95160964536075 +- 43.6572591114911 km/s
For estimator etaMLE the Bulk Flow with mocks is: 377.95160964536075 +- 62.09577254832794 km/s


In [386]:
# B = np.sqrt( Bx**2 + By**2 + Bz**2)
# pour l'estimateur wMLE:
# We import the bulk flow results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results/'
Fit_tech2 = 'wMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'

# We load the bulk flow results of the data:
Bx_data_w, By_data_w, Bz_data_w = np.loadtxt(Input_dir + f'Bulk_flow_data_{Fit_tech2}.txt',skiprows=6,usecols=1,max_rows=3)
# we load the error on the Bx,By,Bz estimation for the data:
err_Bx_w, err_By_w ,err_Bz_w = np.loadtxt( Input_dir + f'Bulk_flow_data_{Fit_tech2}.txt', skiprows=6, usecols=2, max_rows=3)

B_data_w = np.sqrt( Bx_data_w**2 + By_data_w**2 + Bz_data_w**2)
# et on fait la propagation des incertitudes sur B:
err_B_w = (np.abs(Bx_data_w*err_Bx_w) + np.abs(By_data_w*err_By_w) + np.abs(Bz_data_w*err_Bz_w)) / B_data_w

# load the file with the mocks values of BF
all_mocks_w = np.loadtxt(Input_dir + f'Bulk_flow_all_mocks_{Fit_tech2}.txt', skiprows=1)

# we extract the values of Bx,By,Bz and the true values of Ux, Uy, Uz for all the mocks:
Bx_mocks_w = all_mocks_w[:, 1]  
By_mocks_w = all_mocks_w[:, 2] 
Bz_mocks_w = all_mocks_w[:, 3]  
Ux_mocks_w = all_mocks_w[:, 4]
Uy_mocks_w = all_mocks_w[:, 5]
Uz_mocks_w = all_mocks_w[:, 6]

Bx_mocks_CRMS_w = np.std(Bx_mocks_w, ddof=1)
By_mocks_CRMS_w = np.std(By_mocks_w, ddof=1)
Bz_mocks_CRMS_w = np.std(Bz_mocks_w, ddof=1)
Ux_mocks_CRMS_w = np.std(Ux_mocks_w, ddof=1)
Uy_mocks_CRMS_w = np.std(Uy_mocks_w, ddof=1)
Uz_mocks_CRMS_w = np.std(Uz_mocks_w, ddof=1)

# 1. Calculez l'amplitude B pour chaque mock
B_mocks_w = np.sqrt(Bx_mocks_w**2 + By_mocks_w**2 + Bz_mocks_w**2)
U_mocks_w = np.sqrt(Ux_mocks_w**2 + Uy_mocks_w**2 + Uz_mocks_w**2)
# 2. L'erreur est l'écart-type de cette distribution
err_B_mocks_opt_w = np.std(B_mocks_w, ddof=1)
err_U_mocks_true_w = np.std(U_mocks_w, ddof=1)

print('For estimator etaMLE the Bulk Flow with propagation formula is:',B_data_w,'+-',err_B_w,'km/s')
print('For estimator etaMLE the Bulk Flow with mocks is:',B_data_w,'+-',err_B_mocks_opt_w,'km/s')

For estimator etaMLE the Bulk Flow with propagation formula is: 386.6625539008687 +- 44.684122235909754 km/s
For estimator etaMLE the Bulk Flow with mocks is: 386.6625539008687 +- 63.264641934454765 km/s


In [387]:
# B = np.sqrt( Bx**2 + By**2 + Bz**2)
# pour l'estimateur tMLE:
# We import the bulk flow results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results/'
Fit_tech3 = 'tMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'

# We load the bulk flow results of the data:
Bx_data_t, By_data_t, Bz_data_t = np.loadtxt(Input_dir + f'Bulk_flow_data_{Fit_tech3}.txt',skiprows=6,usecols=1,max_rows=3)
# we load the error on the Bx,By,Bz estimation for the data:
err_Bx_t, err_By_t ,err_Bz_t = np.loadtxt( Input_dir + f'Bulk_flow_data_{Fit_tech3}.txt', skiprows=6, usecols=2, max_rows=3)

B_data_t = np.sqrt( Bx_data_t**2 + By_data_t**2 + Bz_data_t**2)
# et on fait la propagation des incertitudes sur B:
err_B_t  = (np.abs(Bx_data_t*err_Bx_t) + np.abs(By_data_t*err_By_t) + np.abs(Bz_data_t*err_Bz_t)) / B_data_t

# load the file with the mocks values of BF
all_mocks_t = np.loadtxt(Input_dir + f'Bulk_flow_all_mocks_{Fit_tech3}.txt', skiprows=1)

# we extract the values of Bx,By,Bz and the true values of Ux, Uy, Uz for all the mocks:
Bx_mocks_t = all_mocks_t[:, 1]  
By_mocks_t = all_mocks_t[:, 2] 
Bz_mocks_t = all_mocks_t[:, 3] 
Ux_mocks_t = all_mocks_t[:, 4]
Uy_mocks_t = all_mocks_t[:, 5]
Uz_mocks_t = all_mocks_t[:, 6]

Bx_mocks_CRMS_t = np.std(Bx_mocks_t, ddof=1)
By_mocks_CRMS_t = np.std(By_mocks_t, ddof=1)
Bz_mocks_CRMS_t = np.std(Bz_mocks_t, ddof=1)
Ux_mocks_CRMS_t = np.std(Ux_mocks_t, ddof=1)
Uy_mocks_CRMS_t = np.std(Uy_mocks_t, ddof=1)
Uz_mocks_CRMS_t = np.std(Uz_mocks_t, ddof=1) 

# 1. Calculez l'amplitude B pour chaque mock
B_mocks_t = np.sqrt(Bx_mocks_t**2 + By_mocks_t**2 + Bz_mocks_t**2)
U_mocks_t = np.sqrt(Ux_mocks_t**2 + Uy_mocks_t**2 + Uz_mocks_t**2)
# 2. L'erreur est l'écart-type de cette distribution
err_B_mocks_opt_t = np.std(B_mocks_t, ddof=1)
err_U_mocks_true_t = np.std(U_mocks_t, ddof=1)

print('For estimator etaMLE the Bulk Flow with propagation formula is:',B_data_t,'+-',err_B_t,'km/s')
print('For estimator etaMLE the Bulk Flow with mocks is:',B_data_t,'+-',err_B_mocks_opt_t,'km/s')

For estimator etaMLE the Bulk Flow with propagation formula is: 964.4515275939416 +- 46.619667152339275 km/s
For estimator etaMLE the Bulk Flow with mocks is: 964.4515275939416 +- 109.89569546937354 km/s


In [ ]:
# chi² entre la prédiction de LamdaCDM et les vrais data:   χ2 = Up ​(Cpq ​+ Rpqv​)**-1 Uq**T
# première etape extraire C_pq la matrice de covariance des B_datas:
#B_data, B_MC, Sv_MC, Um_Cov, SX, SY, SZ, SS = bulk_flow_data_calculator('/renoir/fromenti/Documents/data_DESI/combinedpv/Y1/PV_clustering_data_v5_v13.fits', Fit_tech = Fit_tech1)

In [388]:
def chi2(B_measured,B_pq, C_pq):
    
    B_measured_transposée = B_measured.T
    Tot_pq = B_pq + C_pq
    Tot_pq_inv = np.linalg.inv(Tot_pq)
    # ici la matrice B_measured est la matrice des datas
    chi2 = B_measured @ Tot_pq_inv @ B_measured_transposée
    
    return chi2

C_pq_etaMLE = np.array([[ 476.656962 , -85.661819 , 177.999417],  # je l'ai réecrite ici afin de ne pas refaire tourner le bulk_flow_calculator à chaque fois
 [ -85.661819 , 770.201498,  223.915886],
 [ 177.999417  ,223.915886 ,1340.070206]])

B_eta = np.array([Bx_data_eta,By_data_eta,Bz_data_eta]) # la matrice B qui est comme Up mais ici on prend que le bulk flow par simplicité
chi2_norm_eta = chi2(B_eta,B_pq,C_pq_etaMLE) / 3 
  # on normalise le chi2
print('The chi2 of the prediction of the bulk flow normalize versus the data for etaMLE is:',chi2_norm_eta)

The chi2 of the prediction of the bulk flow normalize versus the data for etaMLE is: 4.808686300885391


In [209]:
# chi² entre la prédiction de LamdaCDM et les vrais data:   χ2 = Up ​(Cpq ​+ Rpqv​)**-1 Uq**T
# première etape extraire C_pq la matrice de covariance des B_datas: avec le wMLE estimator
#B_data_w, B_MC, Sv_MC, Um_Cov, SX, SY, SZ, SS = bulk_flow_data_calculator('/renoir/fromenti/Documents/data_DESI/combinedpv/Y1/PV_clustering_data_v5_v13.fits', Fit_tech = Fit_tech2)
#print(Um_Cov)

In [389]:
C_pq_wMLE = np.array([[ 503.852497 , -56.177918 , 206.728807],  # je l'ai réecrite ici afin de ne pas refaire tourner le bulk_flow_calculator à chaque fois
 [ -56.177918  ,723.424403  ,179.833793],
 [1206.728807 , 179.833793 ,1317.999919 ]])

B_w = np.array([Bx_data_w,By_data_w,Bz_data_w]) # la matrice B qui est comme Up mais ici on prend que le bulk flow par simplicité
chi2_norm_w = chi2(B_w,B_pq,C_pq_wMLE) / 3 

print('The chi2 of the prediction od the bulk flow normalize versus the data for wMLE is:',chi2_norm_w)

The chi2 of the prediction od the bulk flow normalize versus the data for wMLE is: 4.882981145276894


## Table of results

In [390]:
print("Table - Bulk Flow Measured from CF4TF Data")
print("="*100)
print("Component     Measured (ηMLE)     CRMS (ΛCDM)     True mocks (ηMLE)    Estimated mocks (ηMLE)")
print("-"*100)
print(f"Bx (km/s)     {Bx_data_eta:6.2f} ± {err_Bx_eta:5.2f}      ± {Bx_CRMS:6.2f}          ± {Ux_mocks_CRMS_eta:6.2f}              ± {Bx_mocks_CRMS_eta:6.2f} ")
print(f"By (km/s)      {By_data_eta:6.1f} ± {err_By_eta:5.1f}      ± {By_CRMS:6.1f}          ± {Uy_mocks_CRMS_eta:6.2f}              ± {By_mocks_CRMS_eta:6.2f} ")
print(f"Bz (km/s)      {Bz_data_eta:6.1f} ± {err_Bz_eta:5.1f}      ± {Bz_CRMS:6.1f}          ± {Uz_mocks_CRMS_eta:6.2f}              ± {Bz_mocks_CRMS_eta:6.2f} ")
print(f"|B| (km/s)     {B_data_eta:6.2f} ± {err_B_eta:5.2f}      ± {sigma_B:6.1f}          ± {err_U_mocks_true_eta:6.2f}              ± {err_B_mocks_opt_eta:6.2f} ")

Table - Bulk Flow Measured from CF4TF Data
Component     Measured (ηMLE)     CRMS (ΛCDM)     True mocks (ηMLE)    Estimated mocks (ηMLE)
----------------------------------------------------------------------------------------------------
Bx (km/s)     -253.47 ± 21.97      ±  73.11          ±  46.50              ±  64.58 
By (km/s)        32.6 ±  26.9      ±   87.4          ±  52.61              ±  71.94 
Bz (km/s)      -278.5 ±  36.1      ±  219.2          ±  98.71              ± 106.52 
|B| (km/s)     377.95 ± 43.66      ±  247.0          ±  58.19              ±  62.10 


In [392]:
print("Table - Bulk Flow Measured from CF4TF Data")
print("="*100)
print("Component     Measured (wMLE)     CRMS (ΛCDM)     True mocks (wMLE)    Estimated mocks (wMLE)")
print("-"*100)
print(f"Bx (km/s)     {Bx_data_w:6.2f} ± {err_Bx_w:5.2f}      ± {Bx_CRMS:6.2f}          ± {Ux_mocks_CRMS_w:6.2f}              ± {Bx_mocks_CRMS_w:6.2f} ")
print(f"By (km/s)      {By_data_w:6.1f} ± {err_By_w:5.1f}      ± {By_CRMS:6.1f}          ± {Uy_mocks_CRMS_w:6.2f}              ± {By_mocks_CRMS_w:6.2f} ")
print(f"Bz (km/s)      {Bz_data_w:6.1f} ± {err_Bz_w:5.1f}      ± {Bz_CRMS:6.1f}          ± {Uz_mocks_CRMS_w:6.2f}              ± {Bz_mocks_CRMS_w:6.2f} ")  
print(f"|B| (km/s)     {B_data_w:6.2f} ± {err_B_w:5.2f}      ± {sigma_B:6.1f}          ± {err_U_mocks_true_w:6.2f}              ± {err_B_mocks_opt_w:6.2f} ")

Table - Bulk Flow Measured from CF4TF Data
Component     Measured (wMLE)     CRMS (ΛCDM)     True mocks (wMLE)    Estimated mocks (wMLE)
----------------------------------------------------------------------------------------------------
Bx (km/s)     -259.67 ± 22.31      ±  73.11          ±  46.50              ±  66.19 
By (km/s)        32.7 ±  27.0      ±   87.4          ±  52.61              ±  73.66 
Bz (km/s)      -284.6 ±  37.3      ±  219.2          ±  98.71              ± 108.83 
|B| (km/s)     386.66 ± 44.68      ±  247.0          ±  58.19              ±  63.26 


In [393]:
print("Table - Bulk Flow Measured from CF4TF Data")
print("="*100)
print("Component     Measured (tMLE)     CRMS (ΛCDM)     True mocks (tMLE)    Estimated mocks (tMLE)")
print("-"*100)
print(f"Bx (km/s)     {Bx_data_t:6.2f} ± {err_Bx_t:5.2f}      ± {Bx_CRMS:6.2f}          ± {Ux_mocks_CRMS_t:6.2f}              ± {Bx_mocks_CRMS_t:6.2f} ")
print(f"By (km/s)      {By_data_t:6.1f} ± {err_By_t:5.1f}      ± {By_CRMS:6.1f}          ± {Uy_mocks_CRMS_t:6.2f}              ± {By_mocks_CRMS_t:6.2f} ")
print(f"Bz (km/s)      {Bz_data_t:6.1f} ± {err_Bz_t:5.1f}      ± {Bz_CRMS:6.1f}          ± {Uz_mocks_CRMS_t:6.2f}              ± {Bz_mocks_CRMS_t:6.2f} ") 
print(f"|B| (km/s)     {B_data_t:6.2f} ± {err_B_t:5.2f}      ± {sigma_B:6.1f}          ± {err_U_mocks_true_t:6.2f}              ± {err_B_mocks_opt_t:6.2f} ")

Table - Bulk Flow Measured from CF4TF Data
Component     Measured (tMLE)     CRMS (ΛCDM)     True mocks (tMLE)    Estimated mocks (tMLE)
----------------------------------------------------------------------------------------------------
Bx (km/s)     -167.35 ± 24.66      ±  73.11          ±  46.50              ±  75.13 
By (km/s)       106.6 ±  30.6      ±   87.4          ±  52.61              ±  81.68 
Bz (km/s)      -943.8 ±  39.8      ±  219.2          ±  98.71              ± 112.02 
|B| (km/s)     964.45 ± 46.62      ±  247.0          ±  58.19              ± 109.90 


## Extraire la prédiction de LambdaCDM sur le shear moment

In [394]:
# We can extract the Shear moment from the R matrix with the jacobian matrix:

V_bkf = [0,0,0] # setting the Bulk flow 
R_shear = R_mean[3:9, 3:9] # we take only the first 3 rows and 3 columns of the R matrix since we want to extract only the Bulk flow and not the shear

np.set_printoptions(precision=6, suppress=True)
print('R_pq part of the matrix with the shear momoent component:'),
for row in R_shear:
    print("  ".join(f"{x:10.4f}" for x in row))

R_pq part of the matrix with the shear momoent component:
    0.1372      0.0056      0.0068      0.0597     -0.0030     -0.0874
    0.0056      0.0386     -0.0007      0.0038      0.0035     -0.0214
    0.0068     -0.0007      0.0945      0.0129      0.0041     -0.0756
    0.0597      0.0038      0.0129      0.1668      0.0048     -0.0522
   -0.0030      0.0035      0.0041      0.0048      0.1100      0.0106
   -0.0874     -0.0214     -0.0756     -0.0522      0.0106      1.2143


In [395]:
# or la compo Qzz est calculer ensuite pour les datas donc on va enlever la ligne lié à cet element qui est en dernière position si on suit l'ordre pris par la g 
R_shear = R_mean[3:8, 3:8] # we take only the first 3 rows and 3 columns of the R matrix since we want to extract only the Bulk flow and not the shear

np.set_printoptions(precision=6, suppress=True)
print('R_pq part of the matrix with the shear momoent component without Qzz:'),
for row in R_shear:
    print("  ".join(f"{x:10.4f}" for x in row))

R_pq part of the matrix with the shear momoent component without Qzz:
    0.1372      0.0056      0.0068      0.0597     -0.0030
    0.0056      0.0386     -0.0007      0.0038      0.0035
    0.0068     -0.0007      0.0945      0.0129      0.0041
    0.0597      0.0038      0.0129      0.1668      0.0048
   -0.0030      0.0035      0.0041      0.0048      0.1100


In [396]:
# la prediction de lambda CDM est : Q = 0 +- sigma_Q
# 2. Les CRMS sont les racines carrées des éléments diagonaux et on les exctrait dans le meme ordre que la matrice g du code lambdaCDM2_corrected.py
CRMS_Qxx = np.sqrt(R_shear[0, 0])  
CRMS_Qxy = np.sqrt(R_shear[1, 1])
CRMS_Qxz = np.sqrt(R_shear[2, 2])
CRMS_Qyy = np.sqrt(R_shear[3, 3])
CRMS_Qyz = np.sqrt(R_shear[4, 4])
CRMS_Qzz =np.sqrt( np.abs(- R_shear[0, 0] - R_shear[3, 3] ) )

print("Prédiction ΛCDM pour les shear moments :")
print(f"Qxx = 0 ± {CRMS_Qxx:.4f} km/s")
print(f"Qyy = 0 ± {CRMS_Qyy:.4f} km/s")
print(f"Qxy = 0 ± {CRMS_Qxy:.4f} km/s")
print(f"Qxz = 0 ± {CRMS_Qxz:.4f} km/s")
print(f"Qyy = 0 ± {CRMS_Qyy:.4f} km/s")
print(f"Qyz = 0 ± {CRMS_Qyz:.4f} km/s")
print(f"Qzz = 0 ± {CRMS_Qzz:.4f} km/s")

Prédiction ΛCDM pour les shear moments :
Qxx = 0 ± 0.3704 km/s
Qyy = 0 ± 0.4084 km/s
Qxy = 0 ± 0.1965 km/s
Qxz = 0 ± 0.3074 km/s
Qyy = 0 ± 0.4084 km/s
Qyz = 0 ± 0.3316 km/s
Qzz = 0 ± 0.5513 km/s


## Extraire les composantes du shear momoent des datas

In [397]:

# pour l'estimateur etaMLE:
# We import the shear moment results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results/'
Fit_tech1 = 'etaMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'

# We load the bulk flow results of the data:
Qxx_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech1}.txt',skiprows=8,usecols=1,max_rows=1)
Qxy_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech1}.txt',skiprows=9,usecols=1,max_rows=1)
Qxz_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech1}.txt',skiprows=10,usecols=1,max_rows=1)
Qyy_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech1}.txt',skiprows=11,usecols=1,max_rows=1)
Qyz_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech1}.txt',skiprows=12,usecols=1,max_rows=1)
Qzz_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech1}.txt',skiprows=13,usecols=1,max_rows=1)

# we load the error on the Bx,By,Bz estimation for the data:
error_Qxx_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech1}.txt', skiprows=8, usecols=2, max_rows=1)
error_Qxy_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech1}.txt', skiprows=9, usecols=2, max_rows=1)
error_Qxz_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech1}.txt', skiprows=10, usecols=2, max_rows=1)
error_Qyy_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech1}.txt', skiprows=11, usecols=2, max_rows=1)
error_Qyz_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech1}.txt', skiprows=12, usecols=2, max_rows=1)
error_Qzz_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech}.txt', skiprows=13, usecols=2, max_rows=1)

all_mocks_eta = np.loadtxt(Input_dir + f'Shear_moment_all_mocks_with_Qzz_{Fit_tech1}_final.txt', skiprows=1)
# we extract the values of Q and the true values of for all the mocks:
Qxx_opt_mocks_eta = np.std(all_mocks_eta[:, 1])
Qxy_opt_mocks_eta = np.std(all_mocks_eta[:, 2])  
Qxz_opt_mocks_eta = np.std(all_mocks_eta[:, 3])
Qyy_opt_mocks_eta = np.std(all_mocks_eta[:, 4])  
Qyz_opt_mocks_eta = np.std(all_mocks_eta[:, 5])  
Qzz_opt_mocks_eta = np.std(all_mocks_eta[:, 6]) 

Qxx_true_mocks_eta = np.std(all_mocks_eta[:,6])
Qxy_true_mocks_eta = np.std(all_mocks_eta[:,7])
Qxz_true_mocks_eta = np.std(all_mocks_eta[:,8])
Qyy_true_mocks_eta = np.std(all_mocks_eta[:,9])
Qyz_true_mocks_eta = np.std(all_mocks_eta[:,10])
Qzz_true_mocks_eta = np.std(all_mocks_eta[:,11])


print('For estimator etaMLE the Qxx component of the shear moment is:',Qxx_data_eta,'+-',error_Qxx_eta,'km/s')
print('For estimator etaMLE the Qyy component of the shear moment is:',Qyy_data_eta,'+-',error_Qyy_eta,'km/s')
print('For estimator etaMLE the Qzz component of the shear moment is:',Qzz_data_eta,'+-',error_Qzz_eta,'km/s')
print('For estimator etaMLE the Qxy component of the shear moment is:',Qxy_data_eta,'+-',error_Qxy_eta,'km/s')
print('For estimator etaMLE the Qxz component of the shear moment is:',Qxz_data_eta,'+-',error_Qxz_eta,'km/s')
print('For estimator etaMLE the Qyz component of the shear moment is:',Qyz_data_eta,'+-',error_Qyz_eta,'km/s')

For estimator etaMLE the Qxx component of the shear moment is: 0.1132 +- 0.1606 km/s
For estimator etaMLE the Qyy component of the shear moment is: -0.815 +- 0.2579 km/s
For estimator etaMLE the Qzz component of the shear moment is: 0.7017 +- 0.3038 km/s
For estimator etaMLE the Qxy component of the shear moment is: -0.0041 +- 0.1529 km/s
For estimator etaMLE the Qxz component of the shear moment is: -0.7799 +- 0.2748 km/s
For estimator etaMLE the Qyz component of the shear moment is: -0.4634 +- 0.281 km/s


In [398]:

# pour l'estimateur wMLE:
# We import the shear moment results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results/'
Fit_tech2 = 'wMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'

# We load the bulk flow results of the data:
Qxx_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=8,usecols=1,max_rows=1)
Qxy_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=9,usecols=1,max_rows=1)
Qxz_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=10,usecols=1,max_rows=1)
Qyy_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=11,usecols=1,max_rows=1)
Qyz_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=12,usecols=1,max_rows=1)
Qzz_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=13,usecols=1,max_rows=1)

# we load the error on the Bx,By,Bz estimation for the data:
error_Qxx_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=8, usecols=2, max_rows=1)
error_Qxy_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=9, usecols=2, max_rows=1)
error_Qxz_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=10, usecols=2, max_rows=1)
error_Qyy_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=11, usecols=2, max_rows=1)
error_Qyz_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=12, usecols=2, max_rows=1)
error_Qzz_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=13, usecols=2, max_rows=1)


all_mocks_w = np.loadtxt(Input_dir + f'Shear_moment_all_mocks_with_Qzz_{Fit_tech2}_final.txt', skiprows=1)
# we extract the values of Q and the true values of for all the mocks:
Qxx_opt_mocks_w = np.std(all_mocks_w[:, 1])
Qxy_opt_mocks_w = np.std(all_mocks_w[:, 2])  
Qxz_opt_mocks_w = np.std(all_mocks_w[:, 3])
Qyy_opt_mocks_w = np.std(all_mocks_w[:, 4])  
Qyz_opt_mocks_w = np.std(all_mocks_w[:, 5])  
Qzz_opt_mocks_w = np.std(all_mocks_w[:, 6]) 

Qxx_true_mocks_w = np.std(all_mocks_w[:,6])
Qxy_true_mocks_w = np.std(all_mocks_w[:,7])
Qxz_true_mocks_w = np.std(all_mocks_w[:,8])
Qyy_true_mocks_w = np.std(all_mocks_w[:,9])
Qyz_true_mocks_w = np.std(all_mocks_w[:,10])
Qzz_true_mocks_w = np.std(all_mocks_w[:,11])


print('For estimator wMLE the Qxx component of the shear moment is:',Qxx_data_w,'+-',error_Qxx_w,'km/s')
print('For estimator wMLE the Qyy component of the shear moment is:',Qyy_data_w,'+-',error_Qyy_w,'km/s')
print('For estimator wMLE the Qzz component of the shear moment is:',Qzz_data_w,'+-',error_Qzz_w,'km/s')
print('For estimator wMLE the Qxy component of the shear moment is:',Qxy_data_w,'+-',error_Qxy_w,'km/s')
print('For estimator wMLE the Qxz component of the shear moment is:',Qxz_data_w,'+-',error_Qxz_w,'km/s')
print('For estimator wMLE the Qyz component of the shear moment is:',Qyz_data_w,'+-',error_Qyz_w,'km/s')

For estimator wMLE the Qxx component of the shear moment is: 0.1302 +- 0.1782 km/s
For estimator wMLE the Qyy component of the shear moment is: -0.8355 +- 0.2666 km/s
For estimator wMLE the Qzz component of the shear moment is: 0.7054 +- 0.3206 km/s
For estimator wMLE the Qxy component of the shear moment is: -0.0078 +- 0.1542 km/s
For estimator wMLE the Qxz component of the shear moment is: -0.7917 +- 0.2879 km/s
For estimator wMLE the Qyz component of the shear moment is: -0.4743 +- 0.2714 km/s


In [399]:
# extraire la matrice C_pq des datas : pour etaMLE
#result_eta = shear_data_calculator('/renoir/fromenti/Documents/data_DESI/combinedpv/Y1/PV_clustering_data_v5_v13.fits', Fit_tech1)
# l'odre de Q_cov est : Qxx, Qxy, Qxz, Qyy, Qyz
#Q_MC_eta   = result_eta["Q_MC"]  # shear moment calculated for plot, errors bars and contours
#Q_cov_eta  = result_eta["Q_cov"] 
#tot_cov_eta = result_eta["Tot_cov"]

#print(Q_cov_eta) # on prend cette matrice pour que le skear moment 
#print(tot_cov_eta)


In [401]:
# extraire la matrice C_pq des datas : pour wMLE
#result_w = shear_data_calculator('/renoir/fromenti/Documents/data_DESI/combinedpv/Y1/PV_clustering_data_v5_v13.fits', Fit_tech2)

#Q_MC_w   = result_w["Q_MC"]  # shear moment calculated for plot, errors bars and contours
#Q_cov_w = result_w["Q_cov"] 
#tot_cov_w = result_w["Tot_cov"]

#print(tot_cov_w)

In [402]:
def chi2(B_measured,B_pq, C_pq):
    
    B_measured_transposée = B_measured.T
    Tot_pq = B_pq + C_pq
    Tot_pq_inv = np.linalg.inv(Tot_pq)
    # ici la matrice B_measured est la matrice des datas
    chi2 = B_measured @ Tot_pq_inv @ B_measured_transposée
    
    return chi2

C_pq_shear_etaMLE = np.array([[ 0.028257 ,-0.000628 ,-0.006035, -0.018957, -0.002586] ,  # je l'ai réecrite ici afin de ne pas refaire tourner le bulk_flow_calculator à chaque fois
 [ -0.000628 , 0.023508 , 0.008119 ,-0.00373  , 0.008842],
 [-0.006035 , 0.008119 , 0.078114,  0.000127, -0.005283],
 [ -0.018957 ,-0.00373 ,  0.000127 , 0.06693  , 0.015829],
 [ -0.002586 , 0.008842, -0.005283 , 0.015829 , 0.071627]])

Q_eta = np.array([Qxx_data_eta,Qyy_data_eta,Qxy_data_eta,Qxz_data_eta,Qyz_data_eta]) # Up sans Qzz car plus simple et en plus le calule
chi2_norm_eta = chi2(Q_eta,R_shear,C_pq_shear_etaMLE) / 5 # on normalise le chi2

print('The chi2 of the prediction of the shear moment normalize versus the data for etaMLE is:',chi2_norm_eta)

The chi2 of the prediction of the shear moment normalize versus the data for etaMLE is: 2.8561589102441003


In [403]:
C_pq_shear_wMLE = np.array([[0.029639 , 0.00009 , -0.004688, -0.01726 , -0.005093] ,  # je l'ai réecrite ici afin de ne pas refaire tourner le bulk_flow_calculator à chaque fois
 [0.00009  , 0.024939  ,0.00788 , -0.004991,  0.010001 ],
 [ -0.004688 , 0.00788  , 0.080467 ,-0.005361 , 0.002664],
 [ -0.01726 , -0.004991 ,-0.005361 , 0.066692,  0.00942 ],
 [ -0.005093 , 0.010001 , 0.002664  ,0.00942 ,  0.075467]])

Q_w = np.array([Qxx_data_w ,Qyy_data_w ,Qxy_data_w ,Qxz_data_w ,Qyz_data_w ]) # la matrice B qui est comme Up mais ici on prend que le bulk flow par simplicité
chi2_norm_w = chi2(Q_w,R_shear,C_pq_shear_wMLE) / 5  # on normalise le chi2

print('The chi2 of the prediction of the shear moment normalize versus the data for wMLE is:',chi2_norm_w)

The chi2 of the prediction of the shear moment normalize versus the data for wMLE is: 2.9820830294161853


In [405]:
R_mean_tot = R_mean[:8, :8] # on enlève la compo Qzz
print(R_mean_tot)

[[ 5344.839663     1.478425 -2604.814337    -0.464762     0.661625
     -9.172138    -0.628887    -0.397375]
 [    1.478425  7634.511013  2827.423463     2.626138     1.1327
      0.7361       1.958475   -15.221925]
 [-2604.814337  2827.423463 48052.274862    28.1253       4.642087
     18.200913    19.082838    -1.1585  ]
 [   -0.464762     2.626138    28.1253       0.137175     0.005562
      0.0068       0.059687    -0.002962]
 [    0.661625     1.1327       4.642087     0.005562     0.0386
     -0.00065      0.003788     0.0035  ]
 [   -9.172138     0.7361      18.200913     0.0068      -0.00065
      0.094525     0.0129       0.004075]
 [   -0.628887     1.958475    19.082838     0.059687     0.003788
      0.0129       0.166775     0.004775]
 [   -0.397375   -15.221925    -1.1585      -0.002962     0.0035
      0.004075     0.004775     0.109987]]


In [434]:
# chi2 of all the bulk flow and the shear moments:

def chi2_lu(B_measured,B_pq, C_pq):
    
    B_measured_transposée = B_measured.T
    Tot_pq = B_pq + C_pq
    Tot_pq_inv = np.linalg.inv(Tot_pq)
    # ici la matrice B_measured est la matrice des datas
    chi2 = B_measured @ Tot_pq_inv @ B_measured_transposée
    
    return chi2


R_mean_tot = R_mean[:8, :8] # on enlève la compo Qzz size (8,8)

C_pq_etaMLE_tot = np.array([[550.801573 , -19.569822 , 106.683908   , 0.568013   , 0.201634  , -2.426933   ,0.141744 ,   0.381829 ],  # je l'ai réecrite ici afin de ne pas refaire tourner le bulk_flow_calculator à chaque fois c'est tot_cov cette fois
 [  -19.569822   ,1113.376599 , 233.959214   , 0.022437    ,0.692426   , 0.231701  , 0.869721 ,  -4.197613],
 [ 106.683908    ,233.959214   ,2060.846074  ,  0.218228   , 0.357902  ,  5.564905    ,3.243416 ,   1.797453],
 [  0.568013   , 0.022437   , 0.218228    ,0.028405  ,  0.001211   ,-0.004039  ,-0.018927  , -0.005048],
 [  0.201634   , 0.692426 ,   0.357902   , 0.001211   , 0.022604  ,  0.003421 ,  -0.004261  ,  0.00833 ],
 [ -2.426933  ,  0.231701   , 5.564905  , -0.004039   , 0.003421  ,  0.078359,   -0.000533  , -0.000373],
 [  0.141744   , 0.869721  ,  3.243416  , -0.018927  , -0.004261 ,  -0.000533    ,0.066985 ,   0.010434],
 [  0.381829  , -4.197613   , 1.797453  , -0.005048 ,   0.00833   , -0.000373   , 0.010434  ,  0.073084]])  # size (8,8)


C_pq_wMLE_tot = np.array([[548.608632,  -88.072764 ,  81.338432 ,   0.468945 ,   0.19248  ,  -2.375878  ,0.413145 ,   0.671231 ],  
 [  -88.072764, 1191.116932  ,109.878514  ,  0.07592    , 0.38751   ,  0.448049  ,0.565176  , -4.591284],
 [  81.338432  ,109.878514, 2051.236626 ,   0.418965  ,  0.19158   ,  5.889013,   2.54551  ,   2.451711],
 [ 0.468945 ,   0.07592  ,   0.418965  ,  0.028465   , 0.001334  ,   -0.005072,   -0.016805 ,  -0.004273],
 [ 0.19248   ,  0.38751   ,  0.19158  ,   0.001334 ,   0.026265 ,   0.005873,   -0.00774   ,  0.011114 ],
 [ -2.375878 ,   0.448049 ,   5.889013 ,  -0.005072  ,  0.005873  ,  0.080864,  -0.003481  ,  0.003639],
 [  0.413145 ,   0.565176 ,   2.54551   , -0.016805,   -0.00774 ,   -0.003481,  0.065308  ,   0.017202 ],
 [  0.671231  , -4.591284 ,   2.451711  , -0.004273  ,  0.011114 ,   0.003639,   0.017202 ,   0.077877]])  # size (8,8)


R_tot_eta = np.array([Bx_data_eta,By_data_eta,Bz_data_eta,Qxx_data_eta,Qxy_data_eta,Qxz_data_eta,Qyy_data_eta,Qyz_data_eta]) # la matrice B qui est comme Up mais ici on prend que le bulk flow par simplicité
chi2_tot_eta = chi2_lu(R_tot_eta,R_mean_tot,C_pq_etaMLE_tot)  
chi2_norm_tot_eta = chi2_lu(R_tot_eta,R_mean_tot,C_pq_etaMLE_tot) / 8  # car 9 composante - une contrainte de trace nulle ou 8 si on enleve juste Qzz

R_tot_w = np.array([Bx_data_w,By_data_w,Bz_data_w,Qxx_data_w,Qxy_data_w,Qxz_data_w,Qyy_data_w,Qyz_data_w]) # la matrice B qui est comme Up mais ici on prend que le bulk flow par simplicité
chi2_tot_w = chi2_lu(R_tot_w,R_mean_tot,C_pq_wMLE_tot) 
chi2_norm_tot_w = chi2_lu(R_tot_w,R_mean_tot,C_pq_wMLE_tot) / 8  # car 9 composante - une contrainte de trace nulle ou 8 si on enleve juste Qzz

print('The chi2 normalized of the prediction versus the data for etaMLE is:',chi2_norm_tot_eta)
print('The chi2 normalized of the prediction versus the data for wMLE is:',chi2_norm_tot_w)


The chi2 normalized of the prediction versus the data for etaMLE is: 3.314415559732046
The chi2 normalized of the prediction versus the data for wMLE is: 3.419084099998916


In [436]:
# p_values calcul:
import math
from scipy.stats import chi2

#calcul analytique 
def p_value_chi2_manual_even_dof(chi2, k):
    """
    Calcul du p-value pour k pair UNIQUEMENT.
    Formule: P(χ² > x) = e^(-x/2) * Σ_{j=0}^{k/2 - 1} (x/2)^j / j!
    """
    if k % 2 != 0:
        raise ValueError("Cette formule fonctionne uniquement pour k pair")
    
    x = chi2
    sum_term = 0
    for j in range(k // 2):
        sum_term += (x/2)**j / math.factorial(j)
    
    p_value = math.exp(-x/2) * sum_term
    return p_value


print('P_value for etaMLE:',p_value_chi2_manual_even_dof((chi2_tot_eta), 8))  
print('P_value for wMLE:',p_value_chi2_manual_even_dof((chi2_tot_w), 8))  

dof = 8
p_value_eta = chi2.sf(chi2_tot_eta, dof)  # sf = survival function = 1 - cdf
p_value_w = chi2.sf(chi2_tot_w, dof)

print(f"\np-value (etaMLE) with scipy = {p_value_eta:.4f}")
print(f"p-value (wMLE) with scipy = {p_value_w:.4f}")

P_value for etaMLE: 0.0008568836054649561
P_value for wMLE: 0.0006143590717932415

p-value (etaMLE) with scipy = 0.0009
p-value (wMLE) with scipy = 0.0006


In [446]:
from astropy.coordinates import SkyCoord
from astropy import units as u

# Créer un SkyCoord en coordonnées cartésiennes galactiques
# Attention : Les coordonnées galactiques utilisent (u, v, w) pour (x, y, z)
bulk_flow_cart_eta = SkyCoord(
    w=Bx_data_eta* u.km / u.s,   # x (point vers le centre galactique)
    u=By_data_eta* u.km / u.s,   # y (pointe vers l=90°)
    v=Bz_data_eta * u.km / u.s,   # z (pointe vers le pôle Nord galactique)
    frame='galactic',
    representation_type='cartesian')

bulk_flow_cart_w = SkyCoord(
    w=Bx_data_w* u.km / u.s,   # x (point vers le centre galactique)
    u=By_data_w* u.km / u.s,   # y (pointe vers l=90°)
    v=Bz_data_w * u.km / u.s,   # z (pointe vers le pôle Nord galactique)
    frame='galactic',
    representation_type='cartesian')

# Convertir en coordonnées sphériques (l, b)
bulk_flow_sph_eta = bulk_flow_cart_eta.spherical
bulk_flow_sph_w = bulk_flow_cart_w.spherical

# Extraire la longitude et latitude galactiques
l_eta = bulk_flow_sph_eta.lon.deg
b_eta = bulk_flow_sph_eta.lat.deg
l_w = bulk_flow_sph_w.lon.deg
b_w = bulk_flow_sph_w.lat.deg

print(f"Longitude galactique l etaMLE = {l_eta:.1f}°")
print(f"Latitude galactique b etaMLE = {b_eta:.1f}°")
print(f"Longitude galactique l wMLE = {l_w:.1f}°")
print(f"Latitude galactique b wMLE = {b_w:.1f}°")

Longitude galactique l etaMLE = 276.7°
Latitude galactique b etaMLE = -42.1°
Longitude galactique l wMLE = 276.5°
Latitude galactique b wMLE = -42.2°


In [444]:
from astropy.coordinates import SkyCoord
from astropy import units as u


def bulk_flow_with_errors(Bx, By, Bz, err_Bx, err_By, err_Bz):
    """
    Calcule la direction du bulk flow avec incertitudes par propagation d'erreurs.
    """
    # Calcul de l'amplitude et des angles centraux
    B_amp = np.sqrt(Bx**2 + By**2 + Bz**2)
    l_rad = np.arctan2(By, Bx)
    b_rad = np.arcsin(Bz / B_amp) if B_amp > 0 else 0
    
    # Conversion en degrés
    l_deg = np.degrees(l_rad)
    b_deg = np.degrees(b_rad)
    if l_deg < 0:
        l_deg += 360
    
    # Propagation des erreurs sur l'amplitude
    if B_amp > 0:
        err_B = np.sqrt((Bx/B_amp * err_Bx)**2 + 
                        (By/B_amp * err_By)**2 + 
                        (Bz/B_amp * err_Bz)**2)
    else:
        err_B = 0
    
    # Propagation des erreurs sur la longitude l
    R_perp = np.sqrt(Bx**2 + By**2)
    if R_perp > 0:
        dl_dBx = -By / R_perp**2
        dl_dBy = Bx / R_perp**2
        err_l_rad = np.sqrt((dl_dBx * err_Bx)**2 + (dl_dBy * err_By)**2)
        err_l_deg = np.degrees(err_l_rad)
    else:
        err_l_deg = 0
    
    # Propagation des erreurs sur la latitude b
    if B_amp > 0 and R_perp > 0:
        db_dBx = -Bx * Bz / (B_amp**2 * R_perp)
        db_dBy = -By * Bz / (B_amp**2 * R_perp)
        db_dBz = R_perp / B_amp**2
        err_b_rad = np.sqrt((db_dBx * err_Bx)**2 + 
                           (db_dBy * err_By)**2 + 
                           (db_dBz * err_Bz)**2)
        err_b_deg = np.degrees(err_b_rad)
    else:
        err_b_deg = 0
    
    return l_deg, err_l_deg, b_deg, err_b_deg, B_amp, err_B

# Utilisation avec vos données etaMLE
l_eta, err_l_eta, b_eta, err_b_eta, B_amp_eta, err_B_eta = bulk_flow_with_errors(
    Bx_data_eta, By_data_eta, Bz_data_eta,
    err_Bx_eta, err_By_eta, err_Bz_eta
)

print("="*60)
print("Bulk Flow etaMLE avec incertitudes")
print("="*60)
print(f"Bx = {Bx_data_eta:.1f} ± {err_Bx_eta:.1f} km/s")
print(f"By = {By_data_eta:.1f} ± {err_By_eta:.1f} km/s")
print(f"Bz = {Bz_data_eta:.1f} ± {err_Bz_eta:.1f} km/s")
print(f"\nAmplitude |B| = {B_amp_eta:.1f} ± {err_B_eta:.1f} km/s")
print(f"Longitude l = {l_eta:.1f}° ± {err_l_eta:.1f}°")
print(f"Latitude b = {b_eta:.1f}° ± {err_b_eta:.1f}°")

Bulk Flow etaMLE avec incertitudes
Bx = -253.5 ± 22.0 km/s
By = 32.6 ± 26.9 km/s
Bz = -278.5 ± 36.1 km/s

Amplitude |B| = 378.0 ± 30.5 km/s
Longitude l = 172.7° ± 6.0°
Latitude b = -47.5° ± 4.4°


In [440]:
print("Table - Bulk Flow and shear moment Measured from CF4TF Data")
print("="*100)
print("Component     Measured (ηMLE)     CRMS (ΛCDM)     True mocks(ηMLE)    Estimated mocks(ηMLE)")
print("-"*100)
print(f"Bx  (km/s)          {Bx_data_eta:6.2f} ± {err_Bx_eta:5.2f}      ± {Bx_CRMS:6.2f}          ± {Ux_mocks_CRMS_eta:6.2f}              ± {Bx_mocks_CRMS_eta:6.2f} ")
print(f"By  (km/s)           {By_data_eta:6.2f} ± {err_By_eta:5.2f}      ± {By_CRMS:6.1f}          ± {Uy_mocks_CRMS_eta:6.2f}              ± {By_mocks_CRMS_eta:6.2f} ")
print(f"Bz  (km/s)          {Bz_data_eta:6.2f} ± {err_Bz_eta:5.2f}      ± {Bz_CRMS:6.1f}          ± {Uz_mocks_CRMS_eta:6.2f}              ± {Bz_mocks_CRMS_eta:6.2f} ")
print(f"|B| (km/s)           {B_data_eta:6.2f} ± {err_B_eta:5.2f}      ± {sigma_B:6.1f}          ± {err_U_mocks_true_eta:6.2f}              ± {err_B_mocks_opt_eta:6.2f} ")
print(f"Qxx (h*km/s/Mpc)     {Qxx_data_eta:6.3f} ± {error_Qxx_eta:5.3f}      ± {CRMS_Qxx:6.3f}          ± {Qxx_true_mocks_eta:6.3f}              ± {Qxx_opt_mocks_eta:6.3f} ")
print(f"Qxy (h*km/s/Mpc)     {Qxy_data_eta:6.3f} ± {error_Qxy_eta:5.3f}      ± {CRMS_Qxy:6.3f}          ± {Qxy_true_mocks_eta:6.3f}              ± {Qxy_opt_mocks_eta:6.3f}")
print(f"Qxz (h*km/s/Mpc)     {Qxz_data_eta:6.3f} ± {error_Qxz_eta:5.3f}      ± {CRMS_Qxz:6.3f}          ± {Qxz_true_mocks_eta:6.3f}              ± {Qxz_opt_mocks_eta:6.3f}")
print(f"Qyy (h*km/s/Mpc)     {Qyy_data_eta:6.3f} ± {error_Qyy_eta:5.3f}      ± {CRMS_Qyy:6.3f}          ± {Qyy_true_mocks_eta:6.3f}              ± {Qyy_opt_mocks_eta:6.3f} ")
print(f"Qyz (h*km/s/Mpc)     {Qyz_data_eta:6.3f} ± {error_Qyz_eta:5.3f}      ± {CRMS_Qyz:6.3f}          ± {Qyz_true_mocks_eta:6.3f}              ± {Qyz_opt_mocks_eta:6.3f}")
print(f"Qzz (h*km/s/Mpc)     {Qzz_data_eta:6.3f} ± {error_Qzz_eta:5.3f}      ± {CRMS_Qzz:6.3f}          ± {Qzz_true_mocks_eta:6.3f}              ± {Qzz_opt_mocks_eta:6.3f} ")
print(f"Chi2                    {chi2_tot_eta:6.3f}           ddof 8                                                           " )   
print(f"Chi2 normalized         {chi2_norm_tot_eta:6.3f}        ")
print(f"P-value                 {p_value_eta:6.5f} "  )



Table - Bulk Flow and shear moment Measured from CF4TF Data
Component     Measured (ηMLE)     CRMS (ΛCDM)     True mocks(ηMLE)    Estimated mocks(ηMLE)
----------------------------------------------------------------------------------------------------
Bx  (km/s)          -253.47 ± 21.97      ±  73.11          ±  46.50              ±  64.58 
By  (km/s)            32.56 ± 26.91      ±   87.4          ±  52.61              ±  71.94 
Bz  (km/s)          -278.46 ± 36.11      ±  219.2          ±  98.71              ± 106.52 
|B| (km/s)           377.95 ± 43.66      ±  247.0          ±  58.19              ±  62.10 
Qxx (h*km/s/Mpc)      0.113 ± 0.161      ±  0.370          ±  0.612              ±  0.449 
Qxy (h*km/s/Mpc)     -0.004 ± 0.153      ±  0.196          ±  0.383              ±  0.292
Qxz (h*km/s/Mpc)     -0.780 ± 0.275      ±  0.307          ±  0.207              ±  0.479
Qyy (h*km/s/Mpc)     -0.815 ± 0.258      ±  0.408          ±  0.370              ±  0.476 
Qyz (h*km/s/Mpc)     

In [439]:
print("Table - Bulk Flow and shear moment Measured from CF4TF Data")
print("="*100)
print("Component     Measured (wMLE)     CRMS (ΛCDM)     True mocks(wMLE)    Estimated mocks(wMLE)")
print("-"*100)
print(f"Bx  (km/s)          {Bx_data_w:6.2f} ± {err_Bx_w:5.2f}      ± {Bx_CRMS:6.2f}          ± {Ux_mocks_CRMS_w:6.2f}              ± {Bx_mocks_CRMS_w:6.2f} ")
print(f"By  (km/s)           {By_data_w:6.2f} ± {err_By_w:5.2f}      ± {By_CRMS:6.1f}          ± {Uy_mocks_CRMS_w:6.2f}              ± {By_mocks_CRMS_w:6.2f} ")
print(f"Bz  (km/s)          {Bz_data_w:6.2f} ± {err_Bz_w:5.2f}      ± {Bz_CRMS:6.1f}          ± {Uz_mocks_CRMS_w:6.2f}              ± {Bz_mocks_CRMS_w:6.2f} ")
print(f"|B| (km/s)           {B_data_w:6.2f} ± {err_B_w:5.2f}      ± {sigma_B:6.1f}          ± {err_U_mocks_true_w:6.2f}              ± {err_B_mocks_opt_eta:6.2f} ")
print(f"Qxx (h*km/s/Mpc)     {Qxx_data_w:6.3f} ± {error_Qxx_w:5.3f}      ± {CRMS_Qxx:6.3f}          ± {Qxx_true_mocks_w:6.3f}              ± {Qxx_opt_mocks_w:6.3f} ")
print(f"Qxy (h*km/s/Mpc)     {Qxy_data_w:6.3f} ± {error_Qxy_w:5.3f}      ± {CRMS_Qxy:6.3f}          ± {Qxy_true_mocks_w:6.3f}              ± {Qxy_opt_mocks_w:6.3f}")
print(f"Qxz (h*km/s/Mpc)     {Qxz_data_w:6.3f} ± {error_Qxz_w:5.3f}      ± {CRMS_Qxz:6.3f}          ± {Qxz_true_mocks_w:6.3f}              ± {Qxz_opt_mocks_w:6.3f}")
print(f"Qyy (h*km/s/Mpc)     {Qyy_data_w:6.3f} ± {error_Qyy_w:5.3f}      ± {CRMS_Qyy:6.3f}          ± {Qyy_true_mocks_w:6.3f}              ± {Qyy_opt_mocks_w:6.3f} ")
print(f"Qyz (h*km/s/Mpc)     {Qyz_data_w:6.3f} ± {error_Qyz_w:5.3f}      ± {CRMS_Qyz:6.3f}          ± {Qyz_true_mocks_w:6.3f}              ± {Qyz_opt_mocks_w:6.3f}")
print(f"Qzz (h*km/s/Mpc)     {Qzz_data_w:6.3f} ± {error_Qzz_w:5.3f}      ± {CRMS_Qzz:6.3f}          ± {Qzz_true_mocks_w:6.3f}              ± {Qzz_opt_mocks_w:6.3f} ")
print(f"Chi2                    {chi2_tot_w:6.3f}           ddof 8                                                           " )  
print(f"Chi2 normalized         {chi2_norm_tot_w:6.3f} "  )
print(f"P-value                 {p_value_w:6.5f} "  )

Table - Bulk Flow and shear moment Measured from CF4TF Data
Component     Measured (wMLE)     CRMS (ΛCDM)     True mocks(wMLE)    Estimated mocks(wMLE)
----------------------------------------------------------------------------------------------------
Bx  (km/s)          -259.67 ± 22.31      ±  73.11          ±  46.50              ±  66.19 
By  (km/s)            32.65 ± 26.99      ±   87.4          ±  52.61              ±  73.66 
Bz  (km/s)          -284.63 ± 37.25      ±  219.2          ±  98.71              ± 108.83 
|B| (km/s)           386.66 ± 44.68      ±  247.0          ±  58.19              ±  62.10 
Qxx (h*km/s/Mpc)      0.130 ± 0.178      ±  0.370          ±  0.630              ±  0.463 
Qxy (h*km/s/Mpc)     -0.008 ± 0.154      ±  0.196          ±  0.383              ±  0.301
Qxz (h*km/s/Mpc)     -0.792 ± 0.288      ±  0.307          ±  0.207              ±  0.494
Qyy (h*km/s/Mpc)     -0.836 ± 0.267      ±  0.408          ±  0.370              ±  0.490 
Qyz (h*km/s/Mpc)     